# 01 — Xây kho tri thức lai: caption + bộ ba RelTR

**Chạy notebook này một lần**, sau đó dùng notebook **02** để fine-tune và chấm điểm.
Kho lai giữ nguyên ngân hàng caption L5A, thêm ngân hàng các bộ ba có thứ tự `(subject, predicate, object)` từ **RelTR Train + Restval**. Không đưa caption hoặc quan hệ Val/Test vào ngân hàng.

Mỗi bộ ba được viết thành chuỗi, ví dụ `person riding horse`, rồi mã hóa thành **một vector CLIP 512 chiều**. Notebook kiểm tra encoder bằng cách mã hóa lại 32 caption cũ, yêu cầu cosine ≥ 0.999, rồi mới trộn hai không gian. Không có train decoder trong notebook 01; không tạo điểm COCO.

Mã mới đã nhúng trong notebook. Cần dữ liệu Karpathy, kho caption cũ và CSV RelTR Train trên Drive. Chọn Colab GPU → sửa đường dẫn nếu cần → Run all. Không mở hai phiên xây cùng kho.


## 1. Google Drive và môi trường xây kho


In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
from google.colab import drive
sys.dont_write_bytecode = True
drive.mount('/content/drive')
_bank_dir = Path('/content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level2/reltr_triplet_bank_v1')
_encoder_contract = _bank_dir/'encoder_contract.json'
_transformers_spec = 'transformers>=4.45,<5'
if _encoder_contract.is_file():
    _saved_encoder = json.loads(_encoder_contract.read_text(encoding='utf-8'))
    _transformers_spec = 'transformers==' + _saved_encoder['transformers']
    print('Tiếp tục kho cũ với Transformers:', _saved_encoder['transformers'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _transformers_spec, 'huggingface_hub', 'tqdm'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU để mã hóa kho.')
if _encoder_contract.is_file() and torch.__version__ != _saved_encoder['torch']:
    raise RuntimeError('Torch của runtime khác encoder_contract.json (' +
        str(torch.__version__) + ' != ' + str(_saved_encoder['torch']) +
        '). Chưa ghi thêm vào kho; cần runtime/phiên bản Torch khớp để tiếp tục.')


## 2. Đường dẫn và ngưỡng lọc
Kho mới nằm trong `knowledge_base_level2/reltr_triplet_bank_v1`; cả hai notebook phải dùng cùng `BANK_DIR`. Ngưỡng 0.5 là cấu hình khởi đầu, chưa được chứng minh tối ưu; không đổi sau khi đã mở Test.


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
BANK_PARENT = DATA_ROOT/'knowledge_base_level2'
BANK_DIR = BANK_PARENT/'reltr_triplet_bank_v1'
MIN_CONFIDENCE = 0.5
ENCODE_BATCH_SIZE = 128
ANNOTATIONS_DIR = None  # Chỉ điền khi loader báo có nhiều nguồn Karpathy.
KB_DIR = None           # Mặc định knowledge_base_level1_karpathy.
if not DATA_ROOT.is_dir() or not BANK_PARENT.is_dir():
    raise FileNotFoundError('Kiểm tra Drive: cần DataKhoaLuan/Data/knowledge_base_level2 trước khi xây kho.')
print('Đầu vào RelTR:', DATA_ROOT/'reltr_inputs/coco_train_reltr_detection.csv')
print('Kho sẽ lưu:', BANK_DIR)


## 3. Nạp mã xây kho đã nhúng


In [ ]:
EMBEDDED_SOURCES = {'drive_io.py': '"""Verified publication for Colab DriveFS; never depends on remote rename.\n\nStage bytes on local disk, copy/read back a recovery copy, then the destination.\nThis is NOT an atomic transaction or a lock across Colab machines. One writer\nper bank/seed is required. Training checkpoints keep their existing journal.\n"""\nimport errno\nimport hashlib\nimport json\nimport os\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef publish(source, target, attempts=4):\n    """Copy from stable local source; retry transient I/O, verify every byte."""\n    source, target = Path(source), Path(target)\n    expected = sha(source)\n    for attempt in range(attempts):\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            try:\n                dest = target.open(\'r+b\')\n            except FileNotFoundError:\n                dest = target.open(\'xb\')\n            with source.open(\'rb\') as inp, dest:\n                dest.seek(0)\n                for block in iter(lambda: inp.read(8 * 1024 * 1024), b\'\'):\n                    dest.write(block)\n                dest.truncate()\n                dest.flush()\n                try:\n                    os.fsync(dest.fileno())\n                except OSError as exc:\n                    if exc.errno not in {errno.EINVAL, errno.ENOTSUP, errno.ENOSYS}:\n                        raise\n            if sha(target) != expected:\n                raise OSError(errno.EIO, \'Read-back checksum mismatch\', str(target))\n            return expected\n        except OSError as exc:\n            if exc.errno not in {errno.ENOENT, errno.EIO, errno.EAGAIN, errno.EEXIST,\n                                 errno.EBUSY, errno.ETIMEDOUT} or attempt == attempts - 1:\n                raise OSError(f\'Could not publish/verify {target}. Check Drive mount, \'\n                              \'free space and other writers; keep existing outputs.\') from exc\n            time.sleep(.5 * (attempt + 1))\n\n\ndef write_bytes(path, data):\n    path = Path(path)\n    with tempfile.TemporaryDirectory(prefix=\'lv6_publish_\') as folder:\n        local = Path(folder)/\'payload\'\n        local.write_bytes(data)\n        publish(local, path.with_name(path.name + \'.pending\'))\n        publish(local, path)\n\n\ndef write_json(path, value):\n    raw = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode(\'utf-8\')\n    write_bytes(path, raw)\n\n\ndef read_json(path):\n    path = Path(path)\n    try:\n        return json.loads(path.read_text(encoding=\'utf-8\'))\n    except (FileNotFoundError, UnicodeError, json.JSONDecodeError) as original:\n        # Read-only fallback: a later verified write can republish the final file.\n        for suffix in (\'.pending\', \'.tmp\'):\n            try:\n                return json.loads(path.with_name(path.name + suffix).read_text(encoding=\'utf-8\'))\n            except (FileNotFoundError, UnicodeError, json.JSONDecodeError):\n                pass\n        raise original\n\n\ndef json_available(path):\n    path = Path(path)\n    return any(path.with_name(path.name + s).is_file() for s in (\'\', \'.pending\', \'.tmp\'))\n\n\ndef check_output(folder):\n    """Small actual write/read check before expensive Colab work; keep a receipt."""\n    import uuid\n    folder = Path(folder)\n    receipt = folder/\'_io_checks\'/(uuid.uuid4().hex + \'.json\')\n    value = {\'method\': \'local_stage_copy_sha256_v1\', \'folder\': str(folder)}\n    write_json(receipt, value)\n    if read_json(receipt) != value:\n        raise IOError(\'Output preflight failed: \' + str(folder))\n    print(\'Drive write/read OK:\', folder)\n', 'reltr_relation_reranking.py': '"""Lv6 prototype: ordered lexical relation matching, not a scene-graph parser."""\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\n_WN = None\n_LEMMATIZER = None\n_ALIASES = {\n    \'man\': \'person\', \'woman\': \'person\', \'boy\': \'person\', \'girl\': \'person\',\n    \'child\': \'person\', \'people\': \'person\', \'men\': \'person\', \'women\': \'person\',\n    \'children\': \'person\', \'bike\': \'bicycle\', \'bikes\': \'bicycle\',\n    \'bicycles\': \'bicycle\', \'dogs\': \'dog\', \'cats\': \'cat\', \'horses\': \'horse\',\n    \'riding\': \'ride\', \'rides\': \'ride\', \'ridden\': \'ride\', \'rode\': \'ride\',\n    \'chasing\': \'chase\', \'chases\': \'chase\', \'chased\': \'chase\',\n    \'holding\': \'hold\', \'holds\': \'hold\', \'held\': \'hold\',\n    \'wearing\': \'wear\', \'wears\': \'wear\', \'wore\': \'wear\',\n    \'sitting\': \'sit\', \'sits\': \'sit\', \'sat\': \'sit\',\n    \'standing\': \'stand\', \'stands\': \'stand\', \'stood\': \'stand\',\n    \'playing\': \'play\', \'plays\': \'play\', \'played\': \'play\',\n    \'eating\': \'eat\', \'eats\': \'eat\', \'ate\': \'eat\',\n    \'carrying\': \'carry\', \'carries\': \'carry\', \'carried\': \'carry\',\n}\n_PHRASE_ALIASES = {\n    \'next to\': (\'beside\', \'alongside\'),\n    \'above\': (\'over\',), \'below\': (\'under\', \'beneath\'),\n}\n\n\ndef configure_wordnet(enabled=True):\n    """Fail explicitly when requested WordNet data are unavailable."""\n    global _WN, _LEMMATIZER\n    if not enabled:\n        _WN, _LEMMATIZER = None, None\n        return\n    from nltk.corpus import wordnet\n    from nltk.stem import WordNetLemmatizer\n    wordnet.ensure_loaded()\n    _WN, _LEMMATIZER = wordnet, WordNetLemmatizer()\n\n\ndef canonical_image_id(value):\n    text = str(value).strip().replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n    if re.fullmatch(r\'\\d+\', text):\n        return str(int(text))\n    match = re.fullmatch(r\'COCO_(?:train|val|test)2014_(\\d+)\\.jpg\', text, re.I)\n    if match:\n        return str(int(match.group(1)))\n    raise ValueError(f\'image_id/filename COCO không hợp lệ: {value!r}\')\n\n\ndef words(text):\n    result = []\n    for token in re.findall(r"[a-z]+(?:\'[a-z]+)?", str(text).lower().replace(\'_\', \' \')):\n        token = _ALIASES.get(token, token)\n        if _LEMMATIZER is not None:\n            token = _LEMMATIZER.lemmatize(token, pos=\'v\')\n            token = _LEMMATIZER.lemmatize(token, pos=\'n\')\n        result.append(_ALIASES.get(token, token))\n    return tuple(result)\n\n\ndef load_reltr_triplets(csv_path, min_conf=0.5):\n    """Accept real image/prediction/score CSV and split-column CSV; retain empty images."""\n    if not 0 <= min_conf <= 1:\n        raise ValueError(\'min_conf phải nằm trong [0,1]\')\n    table, seen = {}, defaultdict(set)\n    stats = {\'rows\': 0, \'no_relation\': 0, \'below_threshold\': 0, \'duplicates\': 0}\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        reader = csv.DictReader(handle)\n        names = {name.strip().lower(): name for name in (reader.fieldnames or [])}\n        def pick(*options):\n            return next((names[name] for name in options if name in names), None)\n        image_col = pick(\'image_id\', \'image\', \'file_name\')\n        pred_col = pick(\'prediction\')\n        sub_col, rel_col, obj_col = pick(\'subject\', \'s\'), pick(\'predicate\', \'relation\', \'rel\', \'p\'), pick(\'object\', \'o\')\n        score_col = pick(\'score\', \'confidence\', \'conf\', \'reltr_score\')\n        if not image_col or not score_col or not (pred_col or (sub_col and rel_col and obj_col)):\n            raise ValueError(f\'CSV thiếu schema RelTR hỗ trợ: {reader.fieldnames}\')\n        for row_number, row in enumerate(reader, 2):\n            try:\n                image_id = canonical_image_id(row[image_col])\n                table.setdefault(image_id, [])\n                stats[\'rows\'] += 1\n                score = float(row[score_col])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError(\'confidence phải hữu hạn và thuộc [0,1]\')\n                if pred_col:\n                    prediction = row[pred_col].strip()\n                    if prediction.lower() == \'no_relation\':\n                        stats[\'no_relation\'] += 1\n                        continue\n                    parts = re.split(r\'\\s+-\\s+\', prediction, maxsplit=2)\n                    if len(parts) != 3:\n                        raise ValueError(\'prediction phải có dạng subject - relation - object\')\n                    subject, predicate, obj = parts\n                else:\n                    subject, predicate, obj = row[sub_col], row[rel_col], row[obj_col]\n                key = tuple(\' \'.join(x.lower().replace(\'_\', \' \').split()) for x in (subject, predicate, obj))\n                if not all(key):\n                    raise ValueError(\'bộ ba có thành phần rỗng\')\n                if score < min_conf:\n                    stats[\'below_threshold\'] += 1\n                    continue\n                if key in seen[image_id]:\n                    stats[\'duplicates\'] += 1\n                    old = next(t for t in table[image_id] if (t[\'subject\'], t[\'predicate\'], t[\'object\']) == key)\n                    old[\'confidence\'] = max(old[\'confidence\'], score)\n                    continue\n                seen[image_id].add(key)\n                table[image_id].append(dict(zip((\'subject\', \'predicate\', \'object\'), key), confidence=score))\n            except (ValueError, TypeError, KeyError, AttributeError) as exc:\n                raise ValueError(f\'{csv_path}, dòng {row_number}: {exc}\') from exc\n    stats[\'images\'] = len(table)\n    stats[\'triplets\'] = sum(map(len, table.values()))\n    return table, stats\n\n\ndef expand_predicate(predicate, max_synonyms=4):\n    """Sorted same-synset lemmas of first verb sense; no troponym-as-synonym expansion."""\n    base = words(predicate)\n    variants = {base}\n    variants.update(words(p) for p in _PHRASE_ALIASES.get(\' \'.join(base), ()))\n    if _WN is not None and len(base) == 1:\n        synsets = _WN.synsets(base[0], pos=_WN.VERB)\n        if synsets:\n            variants.update(words(name) for name in synsets[0].lemma_names())\n    variants.discard(())\n    return (base,) + tuple(sorted(variants - {base})[:max_synonyms])\n\n\ndef phrase_spans(tokens, phrase):\n    size = len(phrase)\n    return [(i, i + size) for i in range(len(tokens) - size + 1)\n            if size and tokens[i:i + size] == phrase]\n\n\ndef compile_triplets(triplets):\n    return [dict(t, subject_tokens=words(t[\'subject\']), object_tokens=words(t[\'object\']),\n                 predicate_variants=expand_predicate(t[\'predicate\'])) for t in triplets]\n\n\ndef relation_score(text, prepared_triplets, max_gap=12):\n    """Ordered S -> P -> O lexical evidence, restricted to one clause. Not full parsing."""\n    matched, total = [], 0.0\n    clauses = re.split(r\'[.!?;]|\\b(?:and|but|while|whereas)\\b\', text.lower())\n    clauses = [words(clause) for clause in clauses]\n    for triplet in prepared_triplets:\n        conf = triplet[\'confidence\']\n        total += conf\n        hit = False\n        for tokens in clauses:\n            if any(t in {\'not\', \'never\', \'without\', \'no\'} or t.endswith("n\'t") for t in tokens):\n                continue\n            ss = phrase_spans(tokens, triplet[\'subject_tokens\'])\n            oo = phrase_spans(tokens, triplet[\'object_tokens\'])\n            pp = [span for variant in triplet[\'predicate_variants\'] for span in phrase_spans(tokens, variant)]\n            hit = any(s1 <= p0 and p1 <= o0 and p0-s1 <= max_gap and o0-p1 <= max_gap\n                      for s0, s1 in ss for p0, p1 in pp for o0, o1 in oo)\n            if hit:\n                break\n        if hit:\n            matched.append(triplet)\n    return (sum(t[\'confidence\'] for t in matched) / total if total else 0.0), [\n        [t[\'subject\'], t[\'predicate\'], t[\'object\']] for t in matched]\n\n\ndef rerank_candidates(candidates, triplets_by_image, alpha=0.2):\n    """Use original normalized L5A baseline_score; alpha=0 preserves scores and stable ranks."""\n    if not 0 <= alpha <= 1:\n        raise ValueError(\'alpha phải nằm trong [0,1]\')\n    groups, seen = defaultdict(list), set()\n    prepared = {}\n    for raw in candidates:\n        item = dict(raw)\n        image_id = canonical_image_id(item[\'image_id\'])\n        caption_id = str(item[\'caption_id\'])\n        if (image_id, caption_id) in seen:\n            raise ValueError(f\'Caption trùng ID trong cùng ảnh: {image_id}/{caption_id}\')\n        seen.add((image_id, caption_id))\n        if not isinstance(item[\'text\'], str) or not item[\'text\'].strip():\n            raise ValueError(\'Caption rỗng/không phải chuỗi\')\n        base = float(item[\'baseline_score\'])\n        if not math.isfinite(base) or not 0 <= base <= 1:\n            raise ValueError(\'baseline_score phải là điểm L5A đã chuẩn hóa trong [0,1]\')\n        if item.get(\'source_image_id\') is not None and canonical_image_id(item[\'source_image_id\']) == image_id:\n            raise ValueError(\'Ứng viên có cùng source_image_id với ảnh truy vấn\')\n        if image_id not in triplets_by_image:\n            raise ValueError(f\'Thiếu bản ghi RelTR cho ảnh {image_id}; kiểm tra split/image_id\')\n        if image_id not in prepared:\n            prepared[image_id] = compile_triplets(triplets_by_image[image_id]) if alpha else []\n        score, matches = relation_score(item[\'text\'], prepared[image_id]) if alpha else (0.0, [])\n        final = base if alpha == 0 or not prepared[image_id] else (1-alpha)*base + alpha*score\n        item.update(image_id=image_id, baseline_score=base, relation_score=score,\n                    final_score=final, matched_triplets=matches)\n        groups[image_id].append(item)\n    for items in groups.values():\n        items.sort(key=lambda item: -item[\'final_score\'])\n    return dict(groups)\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()', 'hybrid_knowledge.py': '"""Train-only RelTR SPO bank. No caption references from Val/Test are embedded."""\nimport hashlib\nimport importlib.metadata\nimport json\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport reltr_relation_reranking as R\nfrom reltr_relation_reranking import load_reltr_triplets\nimport drive_io as IO\n\n# Exact previous implementation reviewed for this storage-only migration.\n# The encoder, inputs, order, thresholds and recipe must still match exactly.\nLEGACY_STORAGE_CODE = \'d8ded22d3ddf094e2f39662bb613b06fba699cb9e2b9ecdab598b019f4f6340e\'\n\n\ndef compatible_bank_code(saved, current=None):\n    return saved in {current or sha(__file__), LEGACY_STORAGE_CODE}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    separators=(\',\', \':\')).encode(\'utf-8\')).hexdigest()\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return IO.read_json(path)\n\n\ndef write_json(path, value):\n    IO.write_json(path, value)\n\n\ndef guard(path, value):\n    if IO.json_available(path):\n        if read_json(path) != value:\n            raise ValueError(\'Existing configuration differs; do not overwrite: \' + str(path))\n    else:\n        write_json(path, value)\n\n\ndef save_array(path, array):\n    path = Path(path)\n    with tempfile.TemporaryDirectory(prefix=\'lv6_array_\') as folder:\n        local = Path(folder)/\'payload.npy\'\n        with local.open(\'wb\') as f:\n            np.save(f, array, allow_pickle=False)\n        checksum = IO.publish(local, path)\n    write_json(path.with_suffix(\'.sha256.json\'), {\'sha256\': checksum})\n\n\ndef bank_source_contract(path, expected):\n    """Preserve the original contract and chunks for the known I/O-only fix."""\n    if not IO.json_available(path):\n        write_json(path, expected)\n        return expected\n    old = read_json(path)\n    if old == expected:\n        return old\n    candidate = dict(old, source_code=expected[\'source_code\'])\n    if old.get(\'source_code\') != LEGACY_STORAGE_CODE or candidate != expected:\n        raise ValueError(\'Bank inputs/recipe/code changed; preserving existing bank: \' + str(path))\n    print(\'Resume bank from known pre-fix writer; all data/recipe hashes unchanged.\')\n    return old\n\n\ndef load_chunk(path):\n    """None means an uncommitted batch; a checksum mismatch still stops the run."""\n    path = Path(path)\n    if not path.is_file():\n        return None\n    try:\n        return checked_array(path)\n    except (FileNotFoundError, UnicodeError, json.JSONDecodeError):\n        print(\'Re-encode incomplete batch checksum:\', path.name)\n        return None\n\n\ndef checked_array(path):\n    path = Path(path)\n    if sha(path) != read_json(path.with_suffix(\'.sha256.json\'))[\'sha256\']:\n        raise IOError(\'Array checksum failed: \' + str(path))\n    return np.load(path, mmap_mode=\'r\', allow_pickle=False)\n\n\ndef split_ids(root, annotations_dir=None):\n    root = Path(root)\n    candidates = ([Path(annotations_dir)] if annotations_dir else [\n        root/\'COCO_2014/annotations_trainval2014/annotations\',\n        root/\'COCO_2014/annotations\', root/\'annotations\', root])\n    candidates = [p for p in candidates if (p/\'dataset_coco.json\').is_file() or\n                  all((p/f\'captions_karpathy_{s}.json\').is_file() for s in (\'train\', \'val\', \'test\'))]\n    if len(candidates) != 1:\n        raise ValueError(\'Set ANNOTATIONS_DIR to the unique Karpathy source: \' + str(candidates))\n    ann = candidates[0]\n    ids = {s: set() for s in (\'train\', \'val\', \'test\')}\n    if (ann/\'dataset_coco.json\').is_file():\n        for row in read_json(ann/\'dataset_coco.json\')[\'images\']:\n            s = \'train\' if row[\'split\'] == \'restval\' else row[\'split\']\n            iid = int(row[\'cocoid\'])\n            if s not in ids or iid in ids[s]:\n                raise ValueError(\'Invalid split or duplicate image ID\')\n            ids[s].add(iid)\n    else:\n        for s in ids:\n            obj = read_json(ann/f\'captions_karpathy_{s}.json\')\n            ids[s] = ({int(r[\'image_id\']) for r in obj[\'annotations\']} if \'annotations\' in obj\n                      else {int(k) for k in obj})\n    if {s: len(v) for s, v in ids.items()} != {\'train\': 113287, \'val\': 5000, \'test\': 5000}:\n        raise ValueError(\'Wrong Karpathy split sizes\')\n    if ids[\'train\'] & ids[\'val\'] or ids[\'train\'] & ids[\'test\'] or ids[\'val\'] & ids[\'test\']:\n        raise ValueError(\'Split leakage\')\n    return {s: sorted(v) for s, v in ids.items()}\n\n\ndef make_triplet_records(table, train_ids):\n    """Dedup exact ordered SPO; preserve EVERY training source for self-exclusion."""\n    train_ids = set(map(int, train_ids))\n    if set(map(int, table)) != train_ids:\n        raise ValueError(\'RelTR Train CSV must cover exactly Train+Restval, including no_relation rows\')\n    merged = {}\n    for iid, triples in table.items():\n        for t in triples:\n            key = (t[\'subject\'], t[\'predicate\'], t[\'object\'])\n            merged.setdefault(key, {})[int(iid)] = float(t[\'confidence\'])\n    result = []\n    for key in sorted(merged):\n        sources = merged[key]\n        result.append(dict(row_id=len(result), subject=key[0], predicate=key[1], object=key[2],\n            text=\' \'.join(key), source_image_ids=sorted(sources),\n            source_confidences=[sources[i] for i in sorted(sources)]))\n    if not result:\n        raise ValueError(\'No valid training triplets after filtering\')\n    return result\n\n\ndef valid_source(record, query_id):\n    return next((i for i in record[\'source_image_ids\'] if i != int(query_id)), None)\n\n\ndef combine_tokens(caption_rows, caption_scores, relation_rows, relation_scores, offset):\n    """Exactly five unique memory rows; 3 caption + up to 2 SPO, then caption fallback."""\n    if len(caption_rows) != 5 or len(set(map(int, caption_rows))) != 5:\n        raise ValueError(\'Need five distinct caption rows\')\n    n = min(2, len(relation_rows))\n    if len(set(map(int, relation_rows[:n]))) != n or len(relation_scores) < n:\n        raise ValueError(\'Invalid triplet rows\')\n    rows = list(map(int, caption_rows[:3])) + [offset + int(r) for r in relation_rows[:n]]\n    scores = list(map(float, caption_scores[:3])) + list(map(float, relation_scores[:n]))\n    rows += list(map(int, caption_rows[3:5-n]))\n    scores += list(map(float, caption_scores[3:5-n]))\n    if len(rows) != 5 or len(set(rows)) != 5 or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid hybrid memory\')\n    if not all(0 <= s <= 1 for s in scores):\n        raise ValueError(\'Memory confidence outside [0,1]\')\n    return rows, scores\n\n\ndef build_bank(root, out, min_confidence=.5, batch_size=128, annotations_dir=None, kb_dir=None):\n    """Encode unique SPO with the SAME frozen CLIP text space as the existing KB."""\n    import torch\n    from transformers import CLIPModel, CLIPTokenizer\n    from huggingface_hub import model_info\n    from tqdm.auto import tqdm\n    root, out = Path(root), Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    ids = split_ids(root, annotations_dir)\n    train_set = set(ids[\'train\'])\n    if kb_dir is None:\n        found = [p for p in [root/\'knowledge_base_level1_karpathy\', root/\'knowledge_base\']\n                 if (p/\'text_embeddings_train_restval.npy\').is_file()]\n        if len(found) != 1:\n            raise ValueError(\'Set KB_DIR; ambiguous/missing caption bank\')\n        kb_dir = found[0]\n    kb_dir = Path(kb_dir)\n    caption_path, map_path = kb_dir/\'text_embeddings_train_restval.npy\', kb_dir/\'text_embedding_mapping.jsonl\'\n    emb = np.load(caption_path, mmap_mode=\'r\')\n    captions, caption_ids = [], []\n    with map_path.open(encoding=\'utf-8\') as f:\n        for i, line in enumerate(f):\n            r = json.loads(line)\n            if int(r.get(\'row_id\', i)) != i or r[\'split\'] not in (\'train\', \'restval\') or int(r[\'image_id\']) not in train_set:\n                raise ValueError(\'Caption mapping contains invalid row or held-out source\')\n            captions.append(str(r[\'caption\']).strip())\n            caption_ids.append(int(r[\'image_id\']))\n    if emb.shape != (len(captions), 512) or set(caption_ids) != train_set or not all(captions):\n        raise ValueError(\'Caption bank/mapping mismatch\')\n    for start in range(0, len(emb), 8192):\n        x = emb[start:start+8192]\n        if not np.isfinite(x).all() or not np.allclose(np.linalg.norm(x, axis=1), 1, atol=1e-3):\n            raise ValueError(\'Caption bank must be finite, L2-normalized\')\n    csv_path = root/\'reltr_inputs/coco_train_reltr_detection.csv\'\n    table, stats = load_reltr_triplets(csv_path, min_confidence)\n    records = make_triplet_records(table, ids[\'train\'])\n    del table\n    source_contract = dict(version=\'hybrid-spo-bank-v1\', source_code=sha(__file__),\n        parser_code=sha(R.__file__),\n        split_sha256={s: digest(v) for s, v in ids.items()}, min_confidence=min_confidence,\n        caption_embeddings_sha256=sha(caption_path), caption_mapping_sha256=sha(map_path),\n        reltr_train_sha256=sha(csv_path), mapping_sha256=digest(records),\n        caption_rows=len(emb), triplet_rows=len(records), template=\'subject predicate object\',\n        dedup=\'ordered_exact_spo_all_sources\', embedding_dim=512, batch_size=batch_size)\n    source_contract = bank_source_contract(out/\'source_contract.json\', source_contract)\n    write_json(out/\'io_implementation.json\', dict(source_code=sha(__file__),\n        drive_io_code=sha(IO.__file__), original_source_code=source_contract[\'source_code\']))\n    manifest_path = out/\'manifest.json\'\n    if IO.json_available(manifest_path):\n        manifest, _, _ = load_bank(out)\n        if manifest[\'source\'] != source_contract:\n            raise ValueError(\'Existing bank has different sources\')\n        if not manifest_path.is_file():\n            write_json(manifest_path, manifest)\n        print(\'Bank already complete:\', out)\n        return manifest\n    model_id = \'openai/clip-vit-base-patch32\'\n    encoder_path = out/\'encoder_contract.json\'\n    revision = read_json(encoder_path)[\'revision\'] if IO.json_available(encoder_path) else model_info(model_id).sha\n    encoder = dict(model=model_id, revision=revision,\n        transformers=importlib.metadata.version(\'transformers\'), torch=torch.__version__,\n        encode=\'text_model.pooler_output -> text_projection -> L2, float32\', max_length=77)\n    guard(encoder_path, encoder)\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    model = CLIPModel.from_pretrained(model_id, revision=revision).float().to(device).eval()\n    tokenizer = CLIPTokenizer.from_pretrained(model_id, revision=revision)\n    for p in model.parameters():\n        p.requires_grad_(False)\n\n    @torch.inference_mode()\n    def encode(texts):\n        inputs = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors=\'pt\').to(device)\n        x = model.text_projection(model.text_model(**inputs).pooler_output)\n        x = torch.nn.functional.normalize(x.float(), dim=-1).cpu().numpy()\n        if x.shape != (len(texts), 512) or not np.isfinite(x).all():\n            raise ValueError(\'CLIP returned invalid embeddings\')\n        return x.astype(np.float32)\n\n    # Verify encoder identity empirically before mixing old and new embeddings.\n    probe = np.linspace(0, len(captions)-1, 32, dtype=int)\n    sim = np.sum(encode([captions[i] for i in probe]) * np.asarray(emb[probe]), axis=1)\n    if float(sim.min()) < .999:\n        raise ValueError(f\'CLIP differs from caption KB: min cosine={sim.min():.6f}; do not mix spaces\')\n    write_json(out/\'encoder_alignment.json\', dict(rows=probe.tolist(), cosines=sim.tolist(), minimum=float(sim.min())))\n    chunks = []\n    for start in tqdm(range(0, len(records), batch_size), desc=\'CLIP training SPO\'):\n        path = out/\'embedding_chunks\'/f\'{start:08d}.npy\'\n        # Uncommitted chunk is safe to re-encode: no model training or Test occurs here.\n        x = load_chunk(path)\n        if x is None:\n            x = encode([r[\'text\'] for r in records[start:start+batch_size]])\n            save_array(path, x)\n        if x.shape != (min(batch_size, len(records)-start), 512):\n            raise ValueError(\'Bad embedding chunk\')\n        chunks.append(np.asarray(x))\n    triplet_emb = np.concatenate(chunks)\n    save_array(out/\'triplet_embeddings.npy\', triplet_emb)\n    write_json(out/\'triplet_mapping.json\', records)\n    manifest = dict(source=source_contract, encoder=encoder, stats=stats,\n        alignment_min_cosine=float(sim.min()), complete=True,\n        triplet_embeddings_sha256=sha(out/\'triplet_embeddings.npy\'),\n        triplet_mapping_sha256=sha(out/\'triplet_mapping.json\'))\n    manifest[\'bank_sha256\'] = digest(manifest)\n    write_json(manifest_path, manifest)\n    print(\'BUILD COMPLETE:\', len(records), \'unique ordered SPO;\', len(emb), \'caption rows\')\n    return manifest\n\n\ndef load_bank(out):\n    out = Path(out)\n    manifest = read_json(out/\'manifest.json\')\n    if not manifest.get(\'complete\') or manifest[\'bank_sha256\'] != digest({k: v for k, v in manifest.items() if k != \'bank_sha256\'}):\n        raise ValueError(\'Incomplete/changed bank manifest\')\n    for name, key in [(\'triplet_embeddings.npy\', \'triplet_embeddings_sha256\'),\n                      (\'triplet_mapping.json\', \'triplet_mapping_sha256\')]:\n        if sha(out/name) != manifest[key]:\n            raise ValueError(\'Bank payload checksum failed: \' + name)\n    embeddings = np.load(out/\'triplet_embeddings.npy\', mmap_mode=\'r\')\n    mapping = read_json(out/\'triplet_mapping.json\')\n    if embeddings.shape != (len(mapping), 512) or len(mapping) != manifest[\'source\'][\'triplet_rows\']:\n        raise ValueError(\'Bank shape mismatch\')\n    if not np.isfinite(embeddings).all() or not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-3):\n        raise ValueError(\'Invalid SPO vectors\')\n    keys = set()\n    for i, r in enumerate(mapping):\n        key = (r[\'subject\'], r[\'predicate\'], r[\'object\'])\n        if r[\'row_id\'] != i or key in keys or not r[\'source_image_ids\'] or r[\'text\'] != \' \'.join(key):\n            raise ValueError(\'Invalid SPO mapping\')\n        keys.add(key)\n    return manifest, embeddings, mapping\n'}
RUNTIME_DIR = Path('/content/lv6_hybrid_runtime')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_HASHES = {n: hashlib.sha256(s.encode('utf-8')).hexdigest() for n, s in EMBEDDED_SOURCES.items()}
for name, source in EMBEDDED_SOURCES.items():
    path = RUNTIME_DIR/name
    path.write_text(source, encoding='utf-8')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == SOURCE_HASHES[name]
    sys.modules.pop(Path(name).stem, None)
sys.path.insert(0, str(RUNTIME_DIR))
print('Runtime modules verified:', list(EMBEDDED_SOURCES))


## 4. Xây kho — có resume theo từng batch embedding
Bản sửa 29/09: lưu từ đĩa cục bộ, sao chép và đối chiếu SHA-256; không đổi tên file trên DriveFS. Giữ nguyên BANK_DIR, ngưỡng, batch và phiên bản encoder khi chạy tiếp. Khi có encoder_contract.json, cell cài thư viện tự ghim Transformers theo phiên bản đã lưu và kiểm tra Torch trước khi xây. Batch có checksum hợp lệ được dùng lại; batch chưa ghi xong checksum được mã hóa lại. Chỉ chấp nhận bản mã cũ đã biết với cùng toàn bộ dữ liệu/cấu hình; không bỏ kiểm tra checksum. Không chạy cùng kho ở hai Colab.


In [ ]:
import fcntl
import hybrid_knowledge as H
H.IO.check_output(BANK_DIR)
with Path('/tmp/lv6_build_hybrid_bank.lock').open('w') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    try:
        bank_manifest = H.build_bank(DATA_ROOT, BANK_DIR, min_confidence=MIN_CONFIDENCE,
            batch_size=ENCODE_BATCH_SIZE, annotations_dir=ANNOTATIONS_DIR, kb_dir=KB_DIR)
    finally:
        fcntl.flock(lock, fcntl.LOCK_UN)
# fcntl chỉ khóa runtime hiện tại; mỗi kho chỉ giao một người xây.


## 5. Kiểm tra kho và chuyển sang notebook 02
Số caption lấy từ dữ liệu thật, không tự ép theo số ước lượng trong mô tả. Loader L5A hiện tại kiểm tra 566.747 hàng; mọi sai khác phải được xác minh.


In [ ]:
manifest, vectors, mapping = H.load_bank(BANK_DIR)
print('Caption rows:', manifest['source']['caption_rows'])
print('Triplet rows:', len(mapping), '| shape:', vectors.shape)
print('CLIP alignment minimum:', manifest['alignment_min_cosine'])
print('Bank SHA256:', manifest['bank_sha256'])
print('Thống kê CSV:', manifest['stats'])
for row in mapping[:5]:
    print(row['text'], '| số ảnh nguồn:', len(row['source_image_ids']))
print('Xong. Mở 02_FineTune_SinhCaption_ChamDiem_KhoLai_Colab.ipynb.')
